# TP Jour 3 — Spark SQL Avancé, Rigueur Statistique & Couche Gold

### Objectifs de la séance
1. Diagnostiquer l'**asymétrie (`SKEWNESS`)** de vos variables quantitatives et justifier l'emploi d'indicateurs robustes (**Médiane `PERCENTILE_APPROX`**, **IQR `P75 - P25`**, **P90**) plutôt que la seule moyenne arithmétique.
2. Écrire des requêtes **Spark SQL avancées** mobilisant des CTE (`WITH ...`) et des **fonctions de fenêtrage** (`DENSE_RANK`, `ROW_NUMBER`, cumul de **Pareto**, variation temporelle **`LAG`**).
3. Construire au moins **2 à 3 tables agrégées métier dans `/lakehouse/gold/`** prêtes à alimenter votre Dashboard Streamlit à faible latence.
4. Éviter le *Small Files Problem* sur HDFS et compacter votre table Silver avec `OPTIMIZE`.

In [13]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("J3_Analytics_SQL_Gold")
    .getOrCreate()
)

# TODO : Enregistrer votre table Silver principale sous forme de vue temporaire SQL `v_silver`
SILVER_MAIN_PATH = "/lakehouse/silver/flights_enriched"
spark.read.format("delta").load(SILVER_MAIN_PATH).createOrReplaceTempView("v_silver")
print("Lignes disponibles dans v_silver :", spark.sql("SELECT COUNT(*) FROM v_silver").collect()[0][0])

Lignes disponibles dans v_silver : 5819079


In [25]:
# Affichage de quelques vols pour vérifier les données disponibles dans Silver.
spark.sql("""SELECT airline, arrival_delay, cancelled, diverted FROM v_silver LIMIT 20""").show(20, truncate=False)


+-------+-------------+---------+--------+
|airline|arrival_delay|cancelled|diverted|
+-------+-------------+---------+--------+
|DL     |-10          |0        |0       |
|UA     |30           |0        |0       |
|MQ     |-4           |0        |0       |
|OO     |-11          |0        |0       |
|DL     |-15          |0        |0       |
|DL     |-16          |0        |0       |
|EV     |-4           |0        |0       |
|EV     |7            |0        |0       |
|EV     |101          |0        |0       |
|EV     |-20          |0        |0       |
|HA     |4            |0        |0       |
|HA     |6            |0        |0       |
|AA     |-16          |0        |0       |
|AA     |0            |0        |0       |
|OO     |-25          |0        |0       |
|OO     |3            |0        |0       |
|DL     |-11          |0        |0       |
|EV     |1            |0        |0       |
|EV     |-11          |0        |0       |
|AA     |-14          |0        |0       |
+-------+--

In [27]:
# Comparaison des deux COUNT pour voir combien de retards ne sont pas renseignés.
spark.sql("""
    SELECT COUNT(*) AS total_flights, COUNT(arrival_delay) AS known_delays, COUNT(*) - COUNT(arrival_delay) AS missing_delays FROM v_silver
""").show(20, truncate=False)


+-------------+------------+--------------+
|total_flights|known_delays|missing_delays|
+-------------+------------+--------------+
|5819079      |5714008     |105071        |
+-------------+------------+--------------+



## 1. Analyse d'Asymétrie : Moyenne vs Médiane & Écart Interquartile (IQR)

### Exercice 1.1 — Étudier la distribution de votre métrique quantitative principale
Écrivez une requête Spark SQL sur `v_silver` calculant sur votre variable continue principale :
- `COUNT(*)`, `AVG(...)`, `STDDEV(...)`, `SKEWNESS(...)`
- Les quantiles `P25`, `P50` (médiane), `P75`, `P90`, `P95` via `PERCENTILE_APPROX(col, q)` et `MAX(...)`.

Rédigez ensuite en Markdown **votre interprétation statistique** (écart moyenne/médiane, impact des valeurs extrêmes, choix de l'indicateur retenu pour votre dashboard).

In [28]:
# Calcul de la moyenne et de l'écart-type pour observer la dispersion des retards.
spark.sql("""
    SELECT AVG(arrival_delay) AS avg_delay, STDDEV(arrival_delay) AS stddev_delay FROM v_silver
""").show(20, truncate=False)


+-----------------+-----------------+
|avg_delay        |stddev_delay     |
+-----------------+-----------------+
|4.407057357987598|39.27129709388676|
+-----------------+-----------------+



In [17]:
# TODO : Calculer moyenne, écart-type, skewness et quantiles (P25, P50, P75, P90, P95) en Spark SQL
df_dist_summary = spark.sql("""
    SELECT
        COUNT(*) AS total_flights,
        AVG(arrival_delay) AS avg_arrival_delay,
        STDDEV(arrival_delay) AS stddev_arrival_delay,
        SKEWNESS(arrival_delay) AS skewness_arrival_delay,
        PERCENTILE_APPROX(arrival_delay, 0.25) AS p25,
        PERCENTILE_APPROX(arrival_delay, 0.50) AS p50_median,
        PERCENTILE_APPROX(arrival_delay, 0.75) AS p75,
        PERCENTILE_APPROX(arrival_delay, 0.90) AS p90,
        PERCENTILE_APPROX(arrival_delay, 0.95) AS p95,
        MAX(arrival_delay) AS max_arrival_delay
    FROM v_silver
""")
assert df_dist_summary is not None, "TODO : Écrivez la requête SQL de résumé statistique."
df_dist_summary.show(truncate=False)

+-------------+-----------------+--------------------+----------------------+---+----------+---+---+---+-----------------+
|total_flights|avg_arrival_delay|stddev_arrival_delay|skewness_arrival_delay|p25|p50_median|p75|p90|p95|max_arrival_delay|
+-------------+-----------------+--------------------+----------------------+---+----------+---+---+---+-----------------+
|5819079      |4.407057357987598|39.27129709388639   |6.502894504785        |-13|-5        |8  |34 |66 |1971             |
+-------------+-----------------+--------------------+----------------------+---+----------+---+---+---+-----------------+



**Justification statistique :**

Le retard moyen à l'arrivée est de 4,41 minutes, alors que la médiane est de -5 minutes. La médiane est de -5 minutes, ce qui signifie qu'au moins la moitié des vols ayant un retard renseigné arrivent avec 5 minutes d'avance ou davantage. La moyenne est plus élevée à cause des retards importants qui tirent le résultat vers le haut.

Le coefficient `SKEWNESS` de 6,50 confirme une forte asymétrie vers les retards élevés. L'écart-type est de 39,27 minutes, ce qui montre une dispersion importante.

Les quantiles P25 et P75 sont respectivement de -13 et 8 minutes, soit un IQR de 21 minutes. P90 est de 34 minutes et P95 de 66 minutes, tandis que le retard maximal atteint 1 971 minutes.

Pour le dashboard, la médiane et les quantiles sont plus représentatifs des retards habituels que la moyenne seule. La moyenne reste utile comme indicateur complémentaire.

---
## 2. Table Gold #1 : Classement Métier & Loi de Pareto (`OVER ORDER BY ...`)



In [32]:
# Regroupement des vols par compagnie pour comparer leur nombre de vols.

spark.sql("""
    SELECT airline, COUNT(*) AS total_flights FROM v_silver GROUP BY airline ORDER BY total_flights DESC
""").show(20, truncate=False)


+-------+-------------+
|airline|total_flights|
+-------+-------------+
|WN     |1261855      |
|DL     |875881       |
|AA     |725984       |
|OO     |588353       |
|EV     |571977       |
|UA     |515723       |
|MQ     |294632       |
|B6     |267048       |
|US     |198715       |
|AS     |172521       |
|NK     |117379       |
|F9     |90836        |
|HA     |76272        |
|VX     |61903        |
+-------+-------------+



In [33]:
# Utilisation de WITH pour réutiliser les totaux de retard calculés par compagnie.

spark.sql("""
    WITH airline_totals AS (SELECT airline, COUNT(*) AS total_flights, SUM(arrival_delay_minutes) AS total_delay_minutes FROM v_silver GROUP BY airline) SELECT * FROM airline_totals ORDER BY total_delay_minutes DESC
""").show(20, truncate=False)


+-------+-------------+-------------------+
|airline|total_flights|total_delay_minutes|
+-------+-------------+-------------------+
|WN     |1261855      |13849257           |
|AA     |725984       |8611910            |
|DL     |875881       |8046301            |
|EV     |571977       |7504821            |
|UA     |515723       |7301833            |
|OO     |588353       |7215186            |
|MQ     |294632       |4089400            |
|B6     |267048       |3889470            |
|NK     |117379       |2313017            |
|US     |198715       |2091729            |
|F9     |90836        |1698364            |
|AS     |172521       |1284997            |
|VX     |61903        |742936             |
|HA     |76272        |464146             |
+-------+-------------+-------------------+



In [34]:
GOLD_PARETO_PATH = "/lakehouse/gold/airline_delay_pareto"  # TODO : Nommer votre 1re table Gold

df_gold_1 = spark.sql("""
    WITH airline_metrics AS (
        SELECT
            airline,
            COUNT(*) AS total_flights,
            COALESCE(SUM(arrival_delay_minutes), 0) AS total_delay_minutes,
            PERCENTILE_APPROX(arrival_delay_minutes, 0.50) AS median_delay,
            PERCENTILE_APPROX(arrival_delay_minutes, 0.75)
                - PERCENTILE_APPROX(arrival_delay_minutes, 0.25) AS iqr_delay
        FROM v_silver
        GROUP BY airline
    ),
    airline_ranked AS (
        SELECT
            *,
            DENSE_RANK() OVER (ORDER BY total_delay_minutes DESC) AS delay_rank,
            SUM(total_delay_minutes) OVER (
                ORDER BY total_delay_minutes DESC, airline
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS cumulative_delay_minutes,
            SUM(total_delay_minutes) OVER () AS all_delay_minutes
        FROM airline_metrics
    )
    SELECT
        airline, total_flights, total_delay_minutes,
        median_delay, iqr_delay, delay_rank,
        ROUND(100.0 * cumulative_delay_minutes
              / NULLIF(all_delay_minutes, 0), 2) AS cumulative_delay_pct
    FROM airline_ranked
    ORDER BY total_delay_minutes DESC, airline
""")
assert df_gold_1 is not None, "TODO : Écrivez la requête SQL avec fenêtre de Pareto dans `df_gold_1`."

# TODO : Sauvegarder avec .coalesce(1) en Delta dans GOLD_PARETO_PATH
(df_gold_1.coalesce(1).write.format("delta")
 .mode("overwrite").option("overwriteSchema", "true")
 .save(GOLD_PARETO_PATH))
df_gold_1.show(15, truncate=False)

26/11/18 08:56:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 08:56:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 08:56:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 08:56:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 08:56:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 08:56:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 0

+-------+-------------+-------------------+------------+---------+----------+--------------------+
|airline|total_flights|total_delay_minutes|median_delay|iqr_delay|delay_rank|cumulative_delay_pct|
+-------+-------------+-------------------+------------+---------+----------+--------------------+
|WN     |1261855      |13849257           |0           |8        |1         |20.04               |
|AA     |725984       |8611910            |0           |7        |2         |32.50               |
|DL     |875881       |8046301            |0           |3        |3         |44.15               |
|EV     |571977       |7504821            |0           |9        |4         |55.01               |
|UA     |515723       |7301833            |0           |9        |5         |65.57               |
|OO     |588353       |7215186            |0           |8        |6         |76.02               |
|MQ     |294632       |4089400            |0           |10       |7         |81.93               |
|B6     |2

26/11/18 08:57:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 08:57:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 08:57:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 08:57:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 08:57:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 08:57:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [35]:
from pyspark.sql import functions as F

# Nombre total de compagnies
total_companies = df_gold_1.count()

# Nombre de compagnies représentant 20 % du total (arrondi au supérieur)
import math
top_20_count = math.ceil(total_companies * 0.20)

# Pourcentage de retard cumulé par ces 20 % de compagnies
result = (
    df_gold_1
    .orderBy(F.desc("total_delay_minutes"), F.asc("airline"))
    .limit(top_20_count)
    .agg(F.sum("total_delay_minutes").alias("top_delay"))
    .first()
)

total_delay = df_gold_1.agg(
    F.sum("total_delay_minutes").alias("total")
).first()["total"]

percentage = (
    100 * result["top_delay"] / total_delay
    if total_delay else 0
)

print(f"Nombre total de compagnies : {total_companies}")
print(f"20 % des compagnies : {top_20_count}")
print(f"Part des retards : {percentage:.2f} %")

Nombre total de compagnies : 14
20 % des compagnies : 3
Part des retards : 44.15 %


### Résultat de la première table Gold

Les compagnies sont classées selon leur total de minutes de retard. La fonction `DENSE_RANK()` attribue leur rang, tandis que `SUM() OVER` calcule le cumul des retards.

WN arrive en première position avec **13 849 257 minutes de retard**, soit **20,04 % du total**. Les six premières compagnies cumulent **76,02 % des minutes de retard**.

Pour vérifier la loi de Pareto (80/20), une analyse a été réalisée sur les **14 compagnies aériennes**. Les trois premières, représentant environ **21,4 % des compagnies**, cumulent seulement **44,15 % des minutes de retard**.

**La règle des 80/20 n'est donc pas vérifiée**, car une petite proportion des compagnies ne représente pas 80 % des retards. Les résultats montrent néanmoins une concentration des retards sur certaines compagnies aériennes.

## 3. Table Gold #2 : Dynamique Temporelle & Croissance avec `LAG()`

### Exercice 3.1 — Série temporelle et taux d'évolution période-sur-période
Agrégez votre table `v_silver` selon son axe temporel (jour, semaine, mois ou année selon votre sujet) et utilisez la fonction de fenêtrage **`LAG(metric, 1) OVER (ORDER BY periode)`** (ou une moyenne glissante `AVG(...) OVER (ORDER BY periode ROWS BETWEEN ... PRECEDING AND CURRENT ROW)`) pour mesurer la dynamique d'évolution.

Sauvegardez le résultat dans `/lakehouse/gold/<nom_gold_2>`.

In [31]:
# Utilisation de LAG pour comparer le nombre de vols avec celui du mois précédent.
spark.sql("""
    WITH monthly AS (SELECT DATE_TRUNC('MONTH', flight_date) AS month_start, COUNT(*) AS total_flights FROM v_silver WHERE flight_date IS NOT NULL GROUP BY DATE_TRUNC('MONTH', flight_date)) SELECT month_start, total_flights, LAG(total_flights, 1) OVER (ORDER BY month_start) AS previous_month_flights FROM monthly ORDER BY month_start
""").show(20, truncate=False)


26/11/18 08:55:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 08:55:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 08:55:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------------------+-------------+----------------------+
|month_start        |total_flights|previous_month_flights|
+-------------------+-------------+----------------------+
|2015-01-01 00:00:00|469968       |NULL                  |
|2015-02-01 00:00:00|429191       |469968                |
|2015-03-01 00:00:00|504312       |429191                |
|2015-04-01 00:00:00|485151       |504312                |
|2015-05-01 00:00:00|496993       |485151                |
|2015-06-01 00:00:00|503897       |496993                |
|2015-07-01 00:00:00|520718       |503897                |
|2015-08-01 00:00:00|510536       |520718                |
|2015-09-01 00:00:00|464946       |510536                |
|2015-10-01 00:00:00|486165       |464946                |
|2015-11-01 00:00:00|467972       |486165                |
|2015-12-01 00:00:00|479230       |467972                |
+-------------------+-------------+----------------------+



26/11/18 08:55:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 08:55:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 08:55:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 08:55:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [22]:
GOLD_TIME_PATH = "/lakehouse/gold/monthly_flight_trends"  # TODO : Nommer votre 2e table Gold temporelle

df_gold_2 = spark.sql("""
    WITH monthly AS (
        SELECT
            DATE_TRUNC('MONTH', flight_date) AS month_start,
            COUNT(*) AS total_flights
        FROM v_silver
        WHERE flight_date IS NOT NULL
        GROUP BY DATE_TRUNC('MONTH', flight_date)
    ),
    monthly_with_previous AS (
        SELECT
            month_start, total_flights,
            LAG(total_flights, 1) OVER (
                ORDER BY month_start
            ) AS previous_month_flights
        FROM monthly
    )
    SELECT
        month_start, total_flights, previous_month_flights,
        ROUND(100.0 * (total_flights - previous_month_flights)
              / NULLIF(previous_month_flights, 0), 2) AS growth_pct
    FROM monthly_with_previous
    ORDER BY month_start
""")
assert df_gold_2 is not None, "TODO : Écrivez la requête SQL temporelle avec LAG() dans `df_gold_2`."

# TODO : Sauvegarder avec .coalesce(1) en Delta dans GOLD_TIME_PATH
(df_gold_2.coalesce(1).write.format("delta")
 .mode("overwrite").option("overwriteSchema", "true")
 .save(GOLD_TIME_PATH))
df_gold_2.show(20, truncate=False)

26/11/18 08:45:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 08:45:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 08:45:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 08:45:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 08:45:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 08:45:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 0

+-------------------+-------------+----------------------+----------+
|month_start        |total_flights|previous_month_flights|growth_pct|
+-------------------+-------------+----------------------+----------+
|2015-01-01 00:00:00|469968       |NULL                  |NULL      |
|2015-02-01 00:00:00|429191       |469968                |-8.68     |
|2015-03-01 00:00:00|504312       |429191                |17.50     |
|2015-04-01 00:00:00|485151       |504312                |-3.80     |
|2015-05-01 00:00:00|496993       |485151                |2.44      |
|2015-06-01 00:00:00|503897       |496993                |1.39      |
|2015-07-01 00:00:00|520718       |503897                |3.34      |
|2015-08-01 00:00:00|510536       |520718                |-1.96     |
|2015-09-01 00:00:00|464946       |510536                |-8.93     |
|2015-10-01 00:00:00|486165       |464946                |4.56      |
|2015-11-01 00:00:00|467972       |486165                |-3.74     |
|2015-12-01 00:00:00

26/11/18 08:45:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 08:45:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 08:45:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 08:45:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


### Résultat de la deuxième table Gold

La fonction `DATE_TRUNC()` permet de regrouper les vols par mois. `LAG()` récupère ensuite le nombre de vols du mois précédent pour calculer son évolution.

En janvier 2015, la variation est NULL puisqu'aucun mois précédent n'est disponible. En février, le nombre de vols passe de 469 968 à 429 191, soit une baisse de 8,68 %. En mars, il remonte à 504 312 vols, soit une hausse de 17,50 %.

Cette requête permet de suivre l'évolution mensuelle de l'activité aérienne sans réaliser de jointure entre les périodes.

## 4. Table Gold #3 (Axe complémentaire libre) & Compaction `OPTIMIZE`

### Exercice 4.1 — Produire un 3e axe d'analyse (`ROW_NUMBER() OVER (PARTITION BY ...)`) et compacter Silver
1. Créez une 3e table Gold répondant à une question spécifique de votre problématique (ex. top sous-catégorie par région/groupe via `ROW_NUMBER() OVER (PARTITION BY groupe ORDER BY ... DESC) = 1`).
2. Exécutez `OPTIMIZE delta.'{SILVER_MAIN_PATH}'` et expliquez pourquoi la multiplication de micro-fichiers pénalise le NameNode HDFS.

In [23]:
# Premier test de ROW_NUMBER pour comprendre le classement avant d'ajouter PARTITION BY.
spark.sql("""
    WITH airline_counts AS (SELECT airline, COUNT(*) AS total_flights FROM v_silver GROUP BY airline) SELECT airline, total_flights, ROW_NUMBER() OVER (ORDER BY total_flights DESC, airline) AS row_number FROM airline_counts ORDER BY row_number
""").show(20, truncate=False)


26/11/18 08:45:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 08:45:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 08:45:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------+-------------+----------+
|airline|total_flights|row_number|
+-------+-------------+----------+
|WN     |1261855      |1         |
|DL     |875881       |2         |
|AA     |725984       |3         |
|OO     |588353       |4         |
|EV     |571977       |5         |
|UA     |515723       |6         |
|MQ     |294632       |7         |
|B6     |267048       |8         |
|US     |198715       |9         |
|AS     |172521       |10        |
|NK     |117379       |11        |
|F9     |90836        |12        |
|HA     |76272        |13        |
|VX     |61903        |14        |
+-------+-------------+----------+



26/11/18 08:45:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 08:45:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 08:45:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 08:45:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [ ]:
# TODO : Créer votre 3e table Gold (optionnel mais recommandé pour enrichir le dashboard) + lancer OPTIMIZE sur Silver
GOLD_RANKING_PATH = "/lakehouse/gold/top_airlines_by_origin"
df_gold_3 = spark.sql("""
    WITH airline_by_origin AS (
        SELECT
            origin_airport, airline, COUNT(*) AS total_flights
        FROM v_silver
        WHERE origin_airport IS NOT NULL
        GROUP BY origin_airport, airline
    ),
    ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY origin_airport
                ORDER BY total_flights DESC, airline
            ) AS airline_position
        FROM airline_by_origin
    )
    SELECT origin_airport, airline, total_flights, airline_position
    FROM ranked
    WHERE airline_position = 1
""")
(df_gold_3.coalesce(1).write.format("delta")
 .mode("overwrite").option("overwriteSchema", "true")
 .save(GOLD_RANKING_PATH))
df_gold_3.show(20, truncate=False)

spark.sql(f"OPTIMIZE delta.`{SILVER_MAIN_PATH}`").show(truncate=False)

{"ts": "2026-11-18 11:36:29.929", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[TABLE_OR_VIEW_NOT_FOUND] The table or view `v_airports` cannot be found. Verify the spelling and correctness of the schema and catalog.\nSearch path: [`system`.`builtin`, `system`.`session`, `spark_catalog`.`default`].\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01", "context": {"errorClass": "TABLE_OR_VIEW_NOT_FOUND"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o40.sql.\n: org.apache.spark.sql.AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `v_airports` cannot be found. Verify the spelling and correctness of the schema and catalog.\nSearch path: [`system`.`builtin`, `system`.`session`, `spark_catalog`.`default`].\nIf you did not qualify the name with a 

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `v_airports` cannot be found. Verify the spelling and correctness of the schema and catalog.
Search path: [`system`.`builtin`, `system`.`session`, `spark_catalog`.`default`].
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 23 pos 13;
'Project ['m.origin_airport, 'c.airport_name, 'c.city, 'c.state, 'c.latitude, 'c.longitude, 'm.total_flights, 'm.avg_arrival_delay, 'm.cancellation_rate]
+- 'Filter ('between('c.latitude, 18, 72) AND 'between('c.longitude, -180, -65))
   +- 'Join Inner, ('m.origin_airport = 'c.origin_airport)
      :- SubqueryAlias m
      :  +- SubqueryAlias airport_metrics
      :     +- SubqueryAlias airport_metrics
      :        +- Aggregate [origin_airport#2596], [origin_airport#2596, count(1) AS total_flights#14616

### Résultat de la troisième table Gold et compaction

La troisième table Gold identifie la compagnie qui réalise le plus de vols au départ de chaque aéroport.

`ROW_NUMBER()` attribue un numéro aux compagnies, et `PARTITION BY origin_airport` permet de recommencer le classement pour chaque aéroport. Le filtre sur le rang 1 conserve uniquement la première compagnie de chaque aéroport.

La commande `OPTIMIZE` a également été exécutée sur la table Silver. Elle permet de regrouper les petits fichiers Delta en fichiers plus volumineux.

Sur HDFS, un grand nombre de petits fichiers augmente les métadonnées à gérer par le NameNode et peut ralentir les opérations de lecture. La compaction permet de limiter ce problème sans modifier les données métier.